<a href="https://colab.research.google.com/github/CPTR295/Sample-LLMs/blob/main/Text_Emmbedding_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Contrastive Learining

In [1]:
from datasets import load_dataset

train_dataset = load_dataset('nyu-mll/glue','mnli',split='train')
train_dataset = train_dataset.select(range(50000))
train_dataset = train_dataset.remove_columns("idx")

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

mnli/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 52.2MB            

mnli/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mnli/validation_matched-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 1.21MB            

mnli/validation_matched-00000-of-00001.p(…): downloading bytes:           |  0.00B            

...

Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating validation_matched split:   0%|          | 0/9815 [00:00<?, ? examples/s]

Generating validation_mismatched split:   0%|          | 0/9832 [00:00<?, ? examples/s]

Generating test_matched split:   0%|          | 0/9796 [00:00<?, ? examples/s]

Generating test_mismatched split:   0%|          | 0/9847 [00:00<?, ? examples/s]

In [2]:
train_dataset[2]

{'premise': 'One of our number will carry out your instructions minutely.',
 'hypothesis': 'A member of my team will execute your orders with immense precision.',
 'label': 0}

In [3]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('bert-base-uncased')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [4]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

/tmp/ipykernel_2640/2479136289.py:1: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator


In [5]:
val_sts = load_dataset('nyu-mll/glue','stsb',split='validation')
evaluator = EmbeddingSimilarityEvaluator(sentences1=val_sts['sentence1'],sentences2=val_sts['sentence2'],scores=[score/5 for score in val_sts['label']],main_similarity='cosine')

stsb/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  502kB            

stsb/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

stsb/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  151kB            

stsb/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

stsb/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  114kB            

stsb/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

In [6]:
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

/tmp/ipykernel_2640/286474690.py:1: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import SentenceTransformerTrainingArguments


In [7]:
args = SentenceTransformerTrainingArguments(
    output_dir = 'base_embedding_model',
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps = 100,
    fp16=True,
    eval_steps = 100,
    logging_steps=100)

In [8]:
#Loss Function
from sentence_transformers import losses

/tmp/ipykernel_2640/1389578002.py:2: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import losses


In [9]:
train_loss = losses.SoftmaxLoss(
    model=embedding_model,
    sentence_embedding_dimension=embedding_model.get_sentence_embedding_dimension(),
    num_labels=3)


/tmp/ipykernel_2640/2254517258.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  sentence_embedding_dimension=embedding_model.get_sentence_embedding_dimension(),


In [10]:
from sentence_transformers.trainer import SentenceTransformerTrainer

/tmp/ipykernel_2640/3373797685.py:1: DeprecationWarning: Importing from 'sentence_transformers.trainer' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.trainer' instead.
  from sentence_transformers.trainer import SentenceTransformerTrainer


In [11]:
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

In [12]:
trainer.train()

dataset = dataset.select_columns(['hypothesis', 'entailment', 'contradiction'])


Step,Training Loss
100,1.071314
200,0.941279
300,0.881238
400,0.840446
500,0.821906
600,0.828983
700,0.820835
800,0.789831
900,0.780874
1000,0.766414


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1563, training_loss=0.8124594905021019, metrics={'train_runtime': 359.0, 'train_samples_per_second': 139.276, 'train_steps_per_second': 4.354, 'total_flos': 0.0, 'train_loss': 0.8124594905021019, 'epoch': 1.0})

In [13]:
evaluator(embedding_model)#pearson_cosine is what we are interested in

{'pearson_cosine': 0.5813219519045206, 'spearman_cosine': 0.6496936440900553}

In [14]:
!pip install mteb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.9/304.9 kB 30.5 MB/s eta 0:00:00


In [15]:
from mteb import MTEB #Massive Text Embedding Benchmark

In [16]:
evaluation = MTEB(tasks=["Banking77Classification"])
results = evaluation.run(embedding_model)
results

/tmp/ipykernel_2640/2605847703.py:1: DeprecationWarning: MTEB is deprecated and will be removed in future versions. Please use the `mteb.evaluate` function instead.
  evaluation = MTEB(tasks=["Banking77Classification"])
/usr/local/lib/python3.13/dist-packages/mteb/models/model_meta.py:954: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimensions = model.get_sentence_embedding_dimension()
/usr/local/lib/python3.13/dist-packages/mteb/models/model_meta.py:916: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embed_dim=model.get_sentence_embedding_dimension(),


AttributeError: 'MTEB' object has no attribute 'tasks'

In [17]:
import mteb

results = mteb.evaluate(
    embedding_model,
    tasks=["Banking77Classification"]
)

print(results)

/usr/local/lib/python3.13/dist-packages/mteb/models/model_meta.py:954: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimensions = model.get_sentence_embedding_dimension()
/usr/local/lib/python3.13/dist-packages/mteb/models/model_meta.py:916: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embed_dim=model.get_sentence_embedding_dimension(),


AttributeError: 'str' object has no attribute 'metadata'

In [18]:
import mteb

tasks = mteb.get_tasks(
    tasks=["Banking77Classification"]
)

print(tasks)

MTEBTasks(Banking77Classification(name='Banking77Classification', languages=['eng']),)


In [19]:
results = mteb.evaluate(
    embedding_model,
    tasks=tasks
)

print(results)

Evaluating tasks:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/mteb/evaluate.py:169: UserWarning: The task 'Banking77Classification' is superseded by 'Banking77Classification.v2'. We recommend using the newer version of the task unless you are running a specific benchmark. See `get_task('Banking77Classification.v2').metadata.description` to get a description of the task and changes.
  task.check_if_dataset_is_superseded()


train.jsonl:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/365k [00:00<?, ?B/s]

model_name='SentenceTransformer based on google-bert/bert-base-uncased' model_revision='86b5e0934494bd15c9632b12f734a8a67f723594' task_results=[TaskResult(task_name=Banking77Classification, main_score=0.62, scores=..., ...)] exceptions=[] experiment_name=None


In [20]:
results.task_results

[TaskResult(task_name=Banking77Classification, main_score=0.62, scores=..., ...)]

## Cosine Similarity Loss

In [21]:
from datasets import Dataset
train_dataset = load_dataset("nyu-mll/glue","mnli",split='train').select(range(50000))
train_dataset = train_dataset.remove_columns("idx")

mapping = {2:0,1:0,0:1}
train_dataset = Dataset.from_dict({
    "sentence1":train_dataset['premise'],
    "sentence2":train_dataset['hypothesis'],
    "label": [mapping[label] for label in train_dataset['label']]
})


In [22]:
val_sts = load_dataset('nyu-mll/glue','stsb',split='validation')
evaluator = EmbeddingSimilarityEvaluator(sentences1=val_sts['sentence1'],sentences2=val_sts['sentence2'],scores=[score/5 for score in val_sts['label']],main_similarity='cosine')

In [23]:
train_loss = losses.CosineSimilarityLoss(model=embedding_model)

In [24]:
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

Step,Training Loss
100,0.176764
200,0.131388
300,0.114549
400,0.097388
500,0.091607
600,0.095458
700,0.096466
800,0.098575
900,0.098310
1000,0.095537


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1563, training_loss=0.11371497351316329, metrics={'train_runtime': 434.4612, 'train_samples_per_second': 115.085, 'train_steps_per_second': 3.598, 'total_flos': 0.0, 'train_loss': 0.11371497351316329, 'epoch': 1.0})

In [25]:
evaluator(embedding_model)

{'pearson_cosine': 0.6665791448345179, 'spearman_cosine': 0.6783933151415562}

## Multiple Negatives Ranking Loss

In [27]:
from tqdm import tqdm
import random

mnli = load_dataset("nyu-mll/glue",'mnli',split='train')
mnli = mnli.select(range(50000))
mnli = mnli.remove_columns('idx')
mnli = mnli.filter(lambda x: True if x['label']==0 else False)

train_dataset = {"anchor":[],"positive":[],"negative":[]}
soft_negatives = mnli['hypothesis']
soft_negatives = list(mnli["hypothesis"])
random.shuffle(soft_negatives)

for row,soft_negative in tqdm(zip(mnli,soft_negatives)):
    train_dataset['anchor'].append(row['premise'])
    train_dataset['positive'].append(row['hypothesis'])
    train_dataset['negative'].append(soft_negative)

train_dataset = Dataset.from_dict(train_dataset)
len(train_dataset)

16875it [00:01, 10312.75it/s]


16875

In [29]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Create an embedding similarity evaluator for stsb
val_sts = load_dataset('nyu-mll/glue', 'stsb', split='validation')
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine"
)

In [31]:
embedding_model = SentenceTransformer('bert-base-uncased')
train_loss = losses.MultipleNegativesRankingLoss(model=embedding_model)
args = SentenceTransformerTrainingArguments(
    output_dir="mnrloss_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [32]:
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
100,0.322243
200,0.102895
300,0.081755
400,0.058721
500,0.067195


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=528, training_loss=0.12342273263317166, metrics={'train_runtime': 172.0002, 'train_samples_per_second': 98.11, 'train_steps_per_second': 3.07, 'total_flos': 0.0, 'train_loss': 0.12342273263317166, 'epoch': 1.0})

In [33]:
evaluator(embedding_model)

{'pearson_cosine': 0.8068932739507393, 'spearman_cosine': 0.8090114515458715}

## Fine-tuning

Supervised

In [34]:
train_dataset = load_dataset("nyu-mll/glue", "mnli", split="train").select(range(50_000))
train_dataset = train_dataset.remove_columns("idx")

# Create an embedding similarity evaluator for stsb
val_sts = load_dataset('nyu-mll/glue', 'stsb', split='validation')
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine"
)

In [35]:
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
train_loss = losses.MultipleNegativesRankingLoss(model=embedding_model)
args = SentenceTransformerTrainingArguments(
    output_dir="finetuned_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1563, training_loss=0.11035651605402287, metrics={'train_runtime': 120.584, 'train_samples_per_second': 414.649, 'train_steps_per_second': 12.962, 'total_flos': 0.0, 'train_loss': 0.11035651605402287, 'epoch': 1.0})

In [36]:
evaluator(embedding_model)

{'pearson_cosine': 0.8492898090077373, 'spearman_cosine': 0.8491274454225178}